In [ ]:
import pathlib
import re
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn import model_selection
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
import csv
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.callbacks import CSVLogger
from tensorflow.keras.layers import GRU
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
import pickle
from sklearn.model_selection import KFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn import preprocessing
import gensim as gensim

# TODO
# training and validation loss
# https://www.pluralsight.com/guides/data-visualization-deep-learning-model-using-matplotlib
# roc auc curve
# https://scikit-learn.org/1.0/auto_examples/model_selection/plot_roc.html
# micro, macro average roc auc curve
# https://scikit-learn.org/stable/auto_examples/model_selection/plot_roc.html

In [ ]:
genders_github_url = 'https://raw.githubusercontent.com/jahangircsebuet/gender-identification-dataset/d908fe2cf08b4637529fb0e7abb6730f0450304f/genders.csv'
posts_github_url = 'https://raw.githubusercontent.com/jahangircsebuet/gender-identification-dataset/d908fe2cf08b4637529fb0e7abb6730f0450304f/posts.csv'

df = pd.read_csv(posts_github_url, delimiter=";")
df.drop(["facebook_profile", "utime","groupId"], axis = 1, inplace = True)
df.head(5)

,id,author,gender,status
0,1,Iftekhar Uddin,M,ল্যাবে গ্রুপমেটের গ্রাফ নিজে একে দিয়েছি অনেক ...
1,2,Shihab Shonglap,M,"এরকম এক রমজান মাসের স্নিগ্ধ বিকেলে, জনৈক বুয়েট..."
2,3,শিহাবুজ্জামান শান্ত,M,শিবিরবিহীন বুয়েট থেকে শিবিরের কেন্দ্রীয় কমিটির...
3,4,Saddam Hossain,M,জনৈক বন্ধু তার বান্ধবীর সাথে দেখা করার আগে নিজ...
4,5,Rizwan Hasan,M,NaN


In [ ]:
("sample size: ", df.index.size)

('sample size: ', 4051)

## Preprocess step 1: Remove null rows

In [ ]:
df = df.dropna(subset=['status', 'gender'])

In [ ]:
("sample size: ", df.index.size)

('sample size: ', 3896)

## Stop word list

In [ ]:
stop_word_file = 'https://raw.githubusercontent.com/jahangircsebuet/gender-identification-dataset/master/stopwords-bn.txt'
sw_df = pd.read_csv(stop_word_file, delimiter="\t", header=None)
stopword_list = sw_df.values.tolist()
len(stopword_list)

398

## Preprocess step 2: Remove non Bangla characters

In [ ]:
author_posts = []
author_genders = []
for index, row in df.iterrows():
  tokens = [re.sub(r'[^\u0980-\u09FF ]+', '', str(row['status']))]
  author_posts.append(" ".join(tokens))
  author_genders.append(row['gender'])
print(len(author_posts))

3896


## Preprocess step 3: Remove stopword

In [ ]:
author_posts_without_sw = []
for post in author_posts:
    tokens_without_sw = [word for word in post.split() if not word in stopword_list]
    author_posts_without_sw.append(" ".join(tokens_without_sw))
len(author_posts_without_sw)

3896

## Data processor

In [ ]:
import numpy as np
import pathlib
import re
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn import model_selection
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from sklearn.linear_model import SGDClassifier
import random
from sklearn.feature_selection import SelectKBest, chi2

class DataProcessor(object):
  def __init__(self):
    self.posts = []
    self.genders = []
    self.count_vectorizer = CountVectorizer(analyzer='word', ngram_range=(2, 2))
    self.tfidf_vectorizer = TfidfVectorizer(max_features=1000, analyzer='word', ngram_range=(2, 2))

  def load_buet_group_data_and_preprocess(self):
    # stop word list
    stop_word_file = 'https://raw.githubusercontent.com/jahangircsebuet/gender-identification-dataset/master/stopwords-bn.txt'
    sw_df = pd.read_csv(stop_word_file, delimiter="\t", header=None)
    stopword_list = sw_df.values.tolist()
    # len(stopword_list)

    # load the training data
    genders_github_url = 'https://raw.githubusercontent.com/jahangircsebuet/gender-identification-dataset/d908fe2cf08b4637529fb0e7abb6730f0450304f/genders.csv'
    posts_github_url = 'https://raw.githubusercontent.com/jahangircsebuet/gender-identification-dataset/d908fe2cf08b4637529fb0e7abb6730f0450304f/posts.csv'

    df = pd.read_csv(posts_github_url, delimiter=";")

    # preprocess step 1: remove unnecessary columns
    df.drop(["facebook_profile", "utime", "groupId"], axis=1, inplace=True)

    # preprocess step 2: remove null rows
    df = df.dropna(subset=['status', 'gender'])

    # preprocess step 3: remove non Bangla characters
    author_posts = []
    author_genders = []
    for index, row in df.iterrows():
        tokens = [re.sub(r'[^\u0980-\u09FF ]+', '', str(row['status']))]
        author_posts.append(" ".join(tokens))
        author_genders.append(row['gender'])

    # preprocess step 4: remove stop words
    author_posts_without_stopword = []
    for post in author_posts:
        tokens_without_stopword = [word for word in post.split() if not word in stopword_list]
        author_posts_without_stopword.append(" ".join(tokens_without_stopword))

    return author_posts_without_stopword, author_genders


  def load_single_profile_data_and_preprocess(self, filepath, gender):
    # load data from filepath
    df = pd.read_csv(filepath)

    # preprocess step 1: remove null rows
    df = df.dropna(subset=['text'])

    author_posts = []
    author_genders = []
    # preprocess step 2: remove non bangla characters
    for index, row in df.iterrows():
      tokens = [re.sub(r'[^\u0980-\u09FF ]+', '', str(row['text']))]
      author_posts.append(" ".join(tokens))
      author_genders.append(gender)

    # preprocess step 3: remove stop words
    author_posts_without_stopword = []
    for post in author_posts:
        tokens_without_stopword = [word for word in post.split() if not word in stopword_list]
        author_posts_without_stopword.append(" ".join(tokens_without_stopword))

    return author_posts_without_stopword, author_genders

  def read_and_preprocess_data_from_other_profiles(self, filepath, gender):
    # reading csv file
    df = pd.read_csv(filepath)

    # preprocess step 1: remove null rows
    df.dropna(subset=["status"], inplace=True)

    author_posts = []
    author_genders = []
    # preprocess step 2: remove non bangla characters
    for index, row in df.iterrows():
      tokens = [re.sub(r'[^\u0980-\u09FF ]+', '', str(row['status']))]
      author_posts.append(" ".join(tokens))
      author_genders.append(gender)

    # preprocess step 3: remove stop words
    author_posts_without_stopword = []
    for post in author_posts:
        tokens_without_stopword = [word for word in post.split() if not word in stopword_list]
        author_posts_without_stopword.append(" ".join(tokens_without_stopword))

    return author_posts_without_stopword, author_genders

In [ ]:
all_posts = []
all_genders = []

## test data loading and preprocessing for buet groups

In [ ]:
data_processor = DataProcessor()
posts, genders = data_processor.load_buet_group_data_and_preprocess()
("group data size: ", len(posts))
all_posts.extend(posts)
all_genders.extend(genders)

## test data loading and preprocessing for single profiles

In [ ]:
chomok_data_url = "https://raw.githubusercontent.com/jahangircsebuet/gender-identification-dataset/master/chomok-500.csv"
feroza_data_url = "https://raw.githubusercontent.com/jahangircsebuet/gender-identification-dataset/master/feroza-360.csv"
rimi_data_url = "https://raw.githubusercontent.com/jahangircsebuet/gender-identification-dataset/master/rimi-500.csv"
shukhon_data_url = "https://raw.githubusercontent.com/jahangircsebuet/gender-identification-dataset/master/shukhon-800.csv"
jesmin_data_url = "https://raw.githubusercontent.com/jahangircsebuet/gender-identification-dataset/master/jesmin-1000.csv"
fuad_data_url = "https://raw.githubusercontent.com/jahangircsebuet/gender-identification-dataset/master/fuad-338.csv"
lutfor_data_url ="https://raw.githubusercontent.com/jahangircsebuet/gender-identification-dataset/master/lutfor-716.csv"

In [ ]:
data_processor = DataProcessor()
posts, genders = data_processor.load_single_profile_data_and_preprocess(chomok_data_url, "M")
print("chomok data size: ", len(posts))
all_posts.extend(posts)
all_genders.extend(genders)

posts, genders = data_processor.load_single_profile_data_and_preprocess(feroza_data_url, "F")
print("feroza data size: ", len(posts))
all_posts.extend(posts)
all_genders.extend(genders)

posts, genders = data_processor.load_single_profile_data_and_preprocess(rimi_data_url, "F")
print("rimi data size: ", len(posts))
all_posts.extend(posts)
all_genders.extend(genders)

posts, genders = data_processor.load_single_profile_data_and_preprocess(shukhon_data_url, "M")
print("shukhon data size: ", len(posts))
all_posts.extend(posts)
all_genders.extend(genders)

posts, genders = data_processor.load_single_profile_data_and_preprocess(jesmin_data_url, "F")
print("jesmin data size: ", len(posts))
all_posts.extend(posts)
all_genders.extend(genders)

posts, genders = data_processor.load_single_profile_data_and_preprocess(fuad_data_url, "M")
print("fuad data size: ", len(posts))
all_posts.extend(posts)
all_genders.extend(genders)

posts, genders = data_processor.load_single_profile_data_and_preprocess(lutfor_data_url, "M")
print("lutfor data size: ", len(posts))
all_posts.extend(posts)
all_genders.extend(genders)

chomok data size:  487
feroza data size:  293
rimi data size:  408
shukhon data size:  785
jesmin data size:  733
fuad data size:  289
lutfor data size:  603


## load data from other profiles

In [ ]:
total_data_from_other_profile = 0

female_urls = ["/content/shuruve.islam.csv", "/content/sohani.akter.9.csv", "/content/tasnim.rumana.5.csv", "/content/nuzhat.csv",
               "/content/pria.csv", "/content/prianka.csv", "/content/sharmin.jahan.csv", "/content/mahfujaaaa.csv",
               "/content/nil.projapoti.98096.csv", "/content/somi.sky.csv", "/content/afia.raisa.csv", "/content/nasrin.jahan.csv",
               "/content/quazi.sultana.csv", "/content/taufiquea.purna.9.csv", "/content/hamida.barsha.csv", "/content/hena.akter.csv",
               "/content/munira.csv", "/content/rabeya.csv", "/content/soheli.soha.csv", "/content/sopno.ghuri.7127.csv",
               "/content/aroni.ananya.csv", "/content/sonda.tararose.csv", "/content/urbariazi.csv", "/content/jafrinrubayet.csv",
               "/content/jannatulferdous.prity.37.csv", "/content/samiha.tanjim.csv", "/content/nahida.luba.1.csv", "/content/nelufa.yesmin.777.csv",
                "/content/Syeda.happy.zaman.csv", "/content/nnayar.csv", "/content/sigma.sadeq.csv", "/content/anika.tabassum.csv",
               "/content/jarin.alam.714.csv", "/content/jesmin.a.begum.5.csv", "/content/ranashila.ranashila.1.csv", "/content/smrity.dash.csv",
               "/content/suhini.islam.csv", "/content/rifa.sonia.9.csv"]

male_urls = ["/content/chomokh.csv", "/content/chamok.hasan.csv", "/content/jhanker.csv", "/content/ragibhasan.csv",
             "/content/mamun3.14.csv", "/content/superman.com.bd.csv", "/content/DrNiazChowdhury.csv", "/content/aamunir.hasan.csv",
             "/content/Sayed-Sajib.csv", "/content/aswad.csv", "/content/mahdimashrur.matin.csv", "/content/shetubablu.csv",
             "/content/shujon.csv", "/content/gm.faruque.3.csv", "/content/adnan.bashir.fuaad.csv", "/content/anisulhaque.csv",
             "/content/masumbillah.csv", "/content/emon.csv", "/content/imran.csv", "/content/jakaria.csv"]

for url in female_urls:
  posts, genders = data_processor.read_and_preprocess_data_from_other_profiles(url, "F")
  print(url, len(posts))
  total_data_from_other_profile = total_data_from_other_profile + len(posts)
  all_posts.extend(posts)
  all_genders.extend(genders)
total_female_data_from_other_profiles = total_data_from_other_profile

for url in male_urls:
  posts, genders = data_processor.read_and_preprocess_data_from_other_profiles(url, "M")
  total_data_from_other_profile = total_data_from_other_profile + len(posts)
  all_posts.extend(posts)
  all_genders.extend(genders)

("\ntotal data from other profiles: ", total_data_from_other_profile), ("\ntotal female author: ", len(female_urls)), ("\ntotal male author: ", len(male_urls)), ("\ntotal_female_data_from_other_profiles: ", total_female_data_from_other_profiles)

/content/shuruve.islam.csv 203
/content/sohani.akter.9.csv 111
/content/tasnim.rumana.5.csv 146
/content/nuzhat.csv 153
/content/pria.csv 105
/content/prianka.csv 0
/content/sharmin.jahan.csv 9
/content/mahfujaaaa.csv 121
/content/nil.projapoti.98096.csv 89
/content/somi.sky.csv 0
/content/afia.raisa.csv 170
/content/nasrin.jahan.csv 98
/content/quazi.sultana.csv 105
/content/taufiquea.purna.9.csv 113
/content/hamida.barsha.csv 95
/content/hena.akter.csv 0
/content/munira.csv 79
/content/rabeya.csv 60
/content/soheli.soha.csv 114
/content/sopno.ghuri.7127.csv 24
/content/aroni.ananya.csv 96
/content/sonda.tararose.csv 84
/content/urbariazi.csv 98
/content/jafrinrubayet.csv 165
/content/jannatulferdous.prity.37.csv 40
/content/samiha.tanjim.csv 72
/content/nahida.luba.1.csv 0
/content/nelufa.yesmin.777.csv 23
/content/Syeda.happy.zaman.csv 53
/content/nnayar.csv 187
/content/sigma.sadeq.csv 47
/content/anika.tabassum.csv 128
/content/jarin.alam.714.csv 114
/content/jesmin.a.begum.5.csv 

(('\ntotal data from other profiles: ', 7291),
 ('\ntotal female author: ', 38),
 ('\ntotal male author: ', 20),
 ('\ntotal_female_data_from_other_profiles: ', 3320))

In [ ]:
"total data size: ", (3896+487+293+408+785+733+289+603+7291)

('total data size: ', 14785)

In [ ]:
("all_posts.len: ", (len(all_posts), len(all_genders)))

('all_posts.len: ', (14785, 14785))

## Exploratory Data Analysis
### should we filter out author with less than n number of samples?
### should we filter out samples with less than m number of words?
### should we generate wordcloud?
https://www.analyticsvidhya.com/blog/2020/04/beginners-guide-exploratory-data-analysis-text-data/

### check male female ratio

In [ ]:
males = [gender for gender in all_genders if gender == 'M']
females = [gender for gender in all_genders if gender == 'F']
("male: ", len(males)), ("female: ", len(females)), ("males: ", str(100*(len(males)/(len(males)+len(females)))) + "%"), ("females: ", str(100*(len(females)/(len(males)+len(females)))) + "%"),

(('male: ', 9082),
 ('female: ', 5703),
 ('males: ', '61.42712208319242%'),
 ('females: ', '38.57287791680758%'))

## Model Definition

In [ ]:
class ModelTraditionalWithTFIDF():
  def __init__(self, posts, genders, model) -> None:
    self.author_posts = posts
    self.author_genders = genders
    if model == 'GSD':
      self.model = SGDClassifier(max_iter=1000, tol=0.01)
    elif model == 'RF':
      self.model = SGDClassifier(max_iter=1000, tol=0.01)
    elif model == 'NB':
      self.model = MultinomialNB()
    elif model == 'SVM':
      self.model = SVC(kernel='linear')
    elif model == 'LR':
      self.model = SGDClassifier(max_iter=1000, tol=0.01)
    elif model == 'DT':
      self.model = SGDClassifier(max_iter=1000, tol=0.01)
      # self.model = DecisionTreeClassifier(random_state=0)
    elif model == 'KNN':
      self.model = SGDClassifier(max_iter=1000, tol=0.01)
      # self.model = KNeighborsClassifier(n_neighbors=10)

  def train(self):
    label_encoder = LabelEncoder()
    label_encoder.fit(self.author_genders)
    labels = label_encoder.transform(self.author_genders)

    train_x, test_x, train_y, test_y = model_selection.train_test_split(self.author_posts, labels, test_size=0.2)

    male_list = []
    female_list = []

    self.tfidf_vectorizer = TfidfVectorizer(max_features=200, analyzer='word', ngram_range=(1, 1))
    self.tfidf_vectorizer.fit(self.author_posts)

    train_x_tfidf = self.tfidf_vectorizer.transform(train_x)
    test_x_tfidf = self.tfidf_vectorizer.transform(test_x)

    # train svm classifier
    clf = SGDClassifier(max_iter=1000, tol=0.01)
    clf.fit(train_x_tfidf, train_y)
    author_gender_pred = clf.predict(test_x_tfidf)

    accuracy = accuracy_score(test_y, author_gender_pred)
    precision = precision_score(test_y, author_gender_pred)
    recall = recall_score(test_y, author_gender_pred)
    f1 = f1_score(test_y, author_gender_pred)

    return accuracy, precision, recall, f1


In [ ]:
model = ModelTraditionalWithTFIDF(all_posts, all_genders, "SGD")
accuracy, precision, recall, f1 = model.train()
print("SGD", accuracy, precision, recall, f1)
model = ModelTraditionalWithTFIDF(all_posts, all_genders, "NB")
accuracy, precision, recall, f1 = model.train()
print("NB", accuracy, precision, recall, f1)
model = ModelTraditionalWithTFIDF(all_posts, all_genders, "SVM")
accuracy, precision, recall, f1 = model.train()
print("SVM", accuracy, precision, recall, f1)
model = ModelTraditionalWithTFIDF(all_posts, all_genders, "RF")
accuracy, precision, recall, f1 = model.train()
print("RF", accuracy, precision, recall, f1)
model = ModelTraditionalWithTFIDF(all_posts, all_genders, "LR")
accuracy, precision, recall, f1 = model.train()
print("LR", accuracy, precision, recall, f1)
model = ModelTraditionalWithTFIDF(all_posts, all_genders, "DT")
accuracy, precision, recall, f1 = model.train()
print("DT", accuracy, precision, recall, f1)
model = ModelTraditionalWithTFIDF(all_posts, all_genders, "KNN")
accuracy, precision, recall, f1 = model.train()
print("KNN", accuracy, precision, recall, f1)

SGD 0.6723030098072371 0.6788533444121313 0.892896174863388 0.7713004484304933
NB 0.6422049374365911 0.6544303797468355 0.8664804469273742 0.7456730769230769
SVM 0.6665539398038552 0.6872866894197952 0.8642703862660944 0.7656844106463878
RF 0.6618194115657762 0.6605431309904153 0.9168514412416852 0.7678737233054781
LR 0.6628339533310788 0.6648760330578513 0.8963788300835654 0.7634638196915777
DT 0.654041258031789 0.6628595600676819 0.8739542665923034 0.7539090690401731
KNN 0.6594521474467365 0.6594351207531723 0.9020156774916014 0.7618822416646962


## Model DL with TFIDF

In [ ]:
class ModelDLWithTFIDF():
  def __init__(self, posts, genders, model) -> None:
    self.author_posts = posts
    self.author_genders = genders
    self.model=Sequential()
    if model == 'LSTM':
      self.model.add(LSTM(300, return_sequences=False))
    elif model == 'GRU':
      self.model.add(GRU(300, return_sequences=False))

  def train(self):
    labels = np.asarray(pd.get_dummies(self.author_genders))
    train_x, test_x, train_y, test_y = model_selection.train_test_split(self.author_posts, labels, test_size=0.2)

    male_list = []
    female_list = []

    self.tfidf_vectorizer = TfidfVectorizer(max_features=200, analyzer='word', ngram_range=(1, 1))
    self.tfidf_vectorizer.fit(self.author_posts)

    train_x_tfidf = self.tfidf_vectorizer.transform(train_x)
    test_x_tfidf = self.tfidf_vectorizer.transform(test_x)
    train_x_tfidf_dense = train_x_tfidf.toarray()
    test_x_tfidf_dense = test_x_tfidf.toarray()

    train_x_tfidf_dense = train_x_tfidf_dense.reshape(train_x_tfidf_dense.shape[0],1,train_x_tfidf_dense.shape[-1])
    test_x_tfidf_dense = test_x_tfidf_dense.reshape(test_x_tfidf_dense.shape[0],1,test_x_tfidf_dense.shape[-1])

    # print("input shape: ", train_x_tfidf.shape)
    # print("output shape: ", train_y.shape)

    self.model.add(Dense(train_y.shape[1], activation='sigmoid'))

    self.model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    batch, epochs = 32, 5

    history_model = self.model.fit(train_x_tfidf_dense, train_y, batch, epochs, validation_split=0.2)
    scores = self.model.evaluate(test_x_tfidf_dense, test_y, verbose=0)
    author_gender_test_predictions = np.argmax(self.model.predict(test_x_tfidf_dense), axis=1)

    author_gender_original_label = np.argmax(test_y, axis=1)
    accuracy = accuracy_score(author_gender_original_label, author_gender_test_predictions)*100
    f1 = f1_score(author_gender_original_label, author_gender_test_predictions)*100

    # lstm_pre  = precision_score(author_gender_original_label, author_gender_test_predictions) * 100
    # lstm_rec =  recall_score(author_gender_original_label, author_gender_test_predictions) * 100
    # lstm_auc = roc_auc_score(author_gender_original_label, author_gender_test_predictions) * 100
    return accuracy, f1
# model = ModelDLWithTFIDF(all_posts, all_genders, 'LSTM')
# lstm_accuracy, lstm_f1 = model.train()
# model = ModelDLWithTFIDF(all_posts, all_genders, 'GRU')
# gru_accuracy, gru_f1 = model.train()
# ("lstm (acc, f1): ", (lstm_accuracy, lstm_f1)), ("\nlstm (acc, f1): ", (gru_accuracy, gru_f1))

## ModelEmbedding

In [ ]:
class ModelEmbedding(object):
    def __init__(self, cbow=1, cbow_trainable=1, sg=0, sg_trainable=0):
        self.model = Sequential()
        self.cbow = cbow
        self.cbow_trainable = cbow_trainable
        self.sg = sg
        self.sg_trainable = sg_trainable

    def get_stop_words(self):
      stop_word_file = 'https://raw.githubusercontent.com/jahangircsebuet/gender-identification-dataset/master/stopwords-bn.txt'
      sw_df = pd.read_csv(stop_word_file, delimiter="\t", header=None)
      return sw_df.values.tolist()

    def getData(self):
      author_post, author_gender = DataCollection().collectData()
      author_post = [re.sub(r'[^\u0980-\u09FF ]+', '', sentence) for sentence in author_post]
      stop_words = self.get_stop_words()
      author_post_without_sw = []
      for post in author_post:
          tokens_without_sw = [word for word in post.split() if not word in stop_words]
          author_post_without_sw.append(" ".join(tokens_without_sw))
      return author_post_without_sw, author_gender

    def convert_text_to_word_vector(self,author_post,author_gender):
      maxlen = 100
      max_words = 1000

      tokenizer = Tokenizer(num_words=max_words)
      tokenizer.fit_on_texts(author_post)
      sequences = tokenizer.texts_to_sequences(author_post)

      word_index = tokenizer.word_index
      data = pad_sequences(sequences, maxlen=maxlen)
      labels = np.asarray(pd.get_dummies(author_gender)) #
      return data, labels

    def make_dataset(self,X_data, y_data, n_splits):
      X_train_dataset= []
      X_test_dataset = []
      y_train_dataset=  []
      y_test_dataset = []

      for train_index, test_index in KFold(n_splits).split(X_data):
              X_train, X_test = X_data[train_index], X_data[test_index]
              y_train, y_test = y_data[train_index], y_data[test_index]
              X_train_dataset.append(X_train)
              X_test_dataset.append(X_test)
              y_train_dataset.append(y_train)
              y_test_dataset.append(y_test)

      return X_train_dataset, X_test_dataset, y_train_dataset, y_test_dataset

    def get_embedding_vector(self,author_post):
      tokenized_lines = []
      for single_line in author_post:
        tokenized_lines.append(single_line.split())

      if self.cbow == 1:
        # Word2Vec(sentences=common_texts, vector_size=100, window=5, min_count=1, workers=4)
        self.word_model = gensim.models.Word2Vec(tokenized_lines, sg=0, vector_size=300, window=5, min_count=1, workers=4)
      elif self.sg == 1:
        self.word_model = gensim.models.Word2Vec(tokenized_lines, sg=1, vector_size=300, window=5, min_count=1, workers=4)

      embedding_matrix = np.zeros((len(self.word_model.wv.vectors) + 1, 300))  # 29534*300 --> Total word = 29534 ---> prottek word er vector e 300ta element
      for i, vec in enumerate(self.word_model.wv.vectors):
        embedding_matrix[i] = vec

      return embedding_matrix

    def get_model(self, embedding_matrix, author_post_train, author_gender_train):
        model = Sequential()  # 1 * 100 ---
        if self.cbow_trainable == 1:  # 100 * 300  #input_length = 100
            model.add(
                Embedding(len(self.word_model.wv.vectors) + 1, 300, input_length=author_post_train.shape[1], weights=[embedding_matrix],
                          trainable=True))  # size of embedding matrix is 100*300
        elif self.sg_trainable == 1:
            model.add(
                Embedding(len(self.word_model.wv.vectors) + 1, 300, input_length=author_post_train.shape[1], weights=[embedding_matrix],
                          trainable=True))
        else:
            model.add(
                Embedding(len(self.word_model.wv.vectors) + 1, 300, input_length=author_post_train.shape[1], weights=[embedding_matrix],
                          trainable=False))

        model.add(GRU(300, return_sequences=False))
        model.add(Dense(author_gender_train.shape[1], activation="softmax"))
        model.summary()
        model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=['acc'])
        return model

    def process(self):
        # author_post, author_gender = self.getData()
        author_post, author_gender = all_posts, all_genders
        data , labels = self.convert_text_to_word_vector(author_post, author_gender) #size is post_number*100
        X_train_dataset, X_test_dataset, y_train_dataset, y_test_dataset = self.make_dataset(data, labels, 10)
        embedding_matrix = self.get_embedding_vector(author_post) # 29534 * 100 dimension er ekta matrix

        acc_per_fold = []
        loss_per_fold = []
        precision_per_fold = []
        recall_per_fold = []
        f1_score_per_fold = []

        for i in range(len(X_train_dataset)):
            author_post_train = X_train_dataset[i] # 400*100
            author_gender_train = y_train_dataset[i]
            author_post_test = X_test_dataset[i]
            author_gender_test = y_test_dataset[i]
            #batch, epochs = 50, 100
            batch, epochs = 32, 5

            model = self.get_model(embedding_matrix, author_post_train, author_gender_train)
            #csv_logger,tensorboard_callback = self.get_csvlogger_tensorflow_callback()
            history_model = model.fit(author_post_train, author_gender_train, batch, epochs,
                                       validation_split=0.2)
            #, callbacks=[csv_logger, tensorboard_callback])
            scores = model.evaluate(author_post_test, author_gender_test, verbose=0)
            # acc_per_fold.append(scores[1] * 100)
            loss_per_fold.append(scores[0])

            author_gender_test_predictions = model.predict(author_post_test)
            author_gender_predicted_label = np.argmax(author_gender_test_predictions, axis=1)
            author_gender_original_label = np.argmax(author_gender_test, axis=1)

            # print(np.array(np.unique(author_gender_predicted_label, return_counts=True)).T)
            # print(np.array(np.unique(author_gender_original_label, return_counts=True)).T)

            TP = tf.math.count_nonzero(author_gender_predicted_label * author_gender_original_label, dtype=tf.float32)
            TN = tf.math.count_nonzero((author_gender_predicted_label - 1) * (author_gender_original_label - 1),dtype=tf.float32)
            FP = tf.math.count_nonzero(author_gender_predicted_label * (author_gender_original_label - 1), dtype=tf.float32)
            FN = tf.math.count_nonzero((author_gender_predicted_label - 1) * author_gender_original_label, dtype=tf.float32)

            # print("True positive: ", TP, "True negative: ", TN, "False positive: ", FP, "False negative: ", FN)
            print("TP, TN, FP, FN")
            print(TP, TN, FP, FN)

            # print("True negative: ", TN)
            # print("False positive: ", FP)
            # print("False negative: ", FN)

            accuracy = ( TP + TN ) / (TP + FP +FP +FN)
            precision = TP / (TP + FP)
            recall = TP / (TP + FN)
            f1 = 2 * precision * recall / (precision + recall)

            # print("accuracy: ", accuracy)
            # print("precision: ", precision)
            # print("recall: ", recall)
            # print("f1 score: ", f1)

            accuracy = accuracy_score(author_gender_original_label, author_gender_predicted_label)
            precision = precision_score(author_gender_original_label, author_gender_predicted_label)
            recall = recall_score(author_gender_original_label, author_gender_predicted_label)
            f1 = f1_score(author_gender_original_label, author_gender_predicted_label)

            acc_per_fold.append(accuracy)
            precision_per_fold.append(precision)
            recall_per_fold.append(recall)
            f1_score_per_fold.append(f1)

        sum_acc = 0
        sum_loss = 0
        sum_precision = 0
        sum_recall = 0
        sum_f1_score = 0

        for i, acc in enumerate(acc_per_fold):
            sum_acc = sum_acc + acc
            sum_loss = sum_loss+ loss_per_fold[i]
            sum_precision = sum_precision + precision_per_fold[i]
            sum_recall = sum_recall + recall_per_fold[i]
            sum_f1_score = sum_f1_score + f1_score_per_fold[i]

        print("average accuracy is ",sum_acc/10.0)
        print("average precision is ", sum_precision /10.0)
        print("average recall is ",sum_recall/10.0)
        print("average f1 score is ", sum_f1_score /10.0)
        return True
# ModelEmbedding().process()

In [ ]:
# model = ModelEmbedding(all_posts, all_genders, 1, 0)
# accuracy, precision, recall, f1 = model.train()
# accuracy, precision, recall, f1

##Model Embedding GRU

In [ ]:
class ModelEmbeddingGRU(object):
    def __init__(self, cbow=1, cbow_trainable=1, sg=0, sg_trainable=0):
        self.model = Sequential()
        self.cbow = cbow
        self.cbow_trainable = cbow_trainable
        self.sg = sg
        self.sg_trainable = sg_trainable
        self.embedding_matrix = None


    def get_model(self, embedding_matrix, author_post_train, author_gender_train):
        model = Sequential()
        if self.cbow_trainable == 1:
            model.add(
                Embedding(len(self.word_model.wv.vocab) + 1, 300, input_length=author_post_train.shape[1], weights=[embedding_matrix],
                          trainable=True))  # size of embedding matrix is 100*300
        elif self.sg_trainable == 1:
            model.add(
                Embedding(len(self.word_model.wv.vocab) + 1, 300, input_length=author_post_train.shape[1], weights=[embedding_matrix],
                          trainable=True))
        else:
            model.add(
                Embedding(len(self.word_model.wv.vocab) + 1, 300, input_length=author_post_train.shape[1], weights=[embedding_matrix],
                          trainable=False))

        model.add(GRU(300, return_sequences=False))
        model.add(Dense(author_gender_train.shape[1], activation="softmax"))
        model.summary()
        model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=['acc'])
        return model

    def get_embedding_vector(self, author_posts):
        tokenized_lines = []
        for single_line in author_posts:
            tokenized_lines.append(single_line.split())

        word_model = None
        if self.cbow == 1:
            word_model = gensim.models.Word2Vec(tokenized_lines, sg=0, vector_size=300, window=5, min_count=1, workers=4)
        elif self.sg == 1:
            word_model = gensim.models.Word2Vec(tokenized_lines, sg=1, vector_size=300, window=5, min_count=1, workers=4)

        samples = []
        for text in author_posts:
          tokens = []
          for word in text.split():
            tokens.append(word_model.wv.get_vector(word))
          samples.append(tokens)
        return samples

    def train(self, author_posts, author_genders):
        feature_matrix = self.get_embedding_vector(author_posts)

        # what are we doing here?
        data = []
        for line in feature_matrix:
          flat_list = []
          for sublist in line:
              for word in sublist:
                  flat_list.append(word)
          data.append(flat_list)
        data = tf.keras.utils.pad_sequences(data)

        lb = preprocessing.LabelEncoder()
        lb.fit(author_genders)
        labels = lb.transform(author_genders)

        X_data = np.asarray(data, dtype=np.float32)
        y_data = np.array(labels)

        X_train_dataset, X_test_dataset, y_train_dataset, y_test_dataset = self.make_dataset(X_data, y_data, 10)

        acc_per_fold = []
        loss_per_fold = []
        precision_per_fold = []
        recall_per_fold = []
        f1_score_per_fold = []

        kf = KFold(n_splits = 10, shuffle = True, random_state= 0)

        for i, (train, test) in enumerate(kf.split(data, labels)):
            author_post_train = data[train]
            author_gender_train = labels[train]
            author_post_test = data[test]
            author_gender_test = labels[test]

            model = self.get_model(feature_matrix, author_post_train, author_gender_train)
            model.fit(author_post_train, author_gender_train)
            y_pred = model.predict(author_post_test)

            # print("training SVM for i: ", i)
            # svc_classifier = SVC(kernel='linear')
            # svc_classifier.fit(author_post_train, author_gender_train)
            # y_pred = svc_classifier.predict(author_post_test)


            accuracy = accuracy_score(author_gender_test, y_pred)
            precision = precision_score(author_gender_test, y_pred)
            recall = recall_score(author_gender_test, y_pred)
            f1 = f1_score(author_gender_test, y_pred)

            acc_per_fold.append(accuracy)
            precision_per_fold.append(precision)
            recall_per_fold.append(recall)
            f1_score_per_fold.append(f1)

        sum_acc = 0
        sum_precision = 0
        sum_recall = 0
        sum_f1_score = 0

        for i, acc in enumerate(acc_per_fold):
            sum_acc = sum_acc + acc
            sum_precision = sum_precision + precision_per_fold[i]
            sum_recall = sum_recall + recall_per_fold[i]
            sum_f1_score = sum_f1_score + f1_score_per_fold[i]

        return sum_acc/10.0, sum_precision /10.0, sum_recall/10.0, sum_f1_score /10.0

model = ModelEmbeddingGRU()
accuracy, precision, recall, f1 = model.train(all_posts, all_genders)
